# Build an AI agent from scratch: the notebook walkthrough

This notebook follows the [dev.to guide](https://dev.to/dextralabs/how-to-build-an-ai-agent-from-scratch-using-claude-api-with-full-code-4b40) **one cell at a time**, so you can see what every step does.

**How to run it** (from the repo root, in a terminal):

```bash
cd backend && source .venv/bin/activate      # the venv from the README
pip install -e ".[notebook]"                 # adds jupyter + python-dotenv
cd .. && jupyter notebook notebooks/walkthrough.ipynb
```

Then press **Shift + Enter** on each cell, top to bottom. Your key is read from `backend/.env`. It is never printed or stored in this notebook.

| Guide step | Cell | In the real app |
|---|---|---|
| 1 · Basic setup | Hello Claude | `backend/app/llm.py` |
| 2 · Tools | Toolbox | `backend/app/tools/` |
| 3 · ReAct loop | Minimal loop | `backend/app/agent.py` |
| 4 · Run it | Real agent | `backend/app/cli.py` |
| 5 · Memory | Two questions | `backend/app/main.py` + Supabase |

In [ ]:
# Setup: make the backend importable and load backend/.env into the environment
import sys, os, json
from pathlib import Path
from dotenv import load_dotenv

BACKEND = Path.cwd().parent / "backend" if Path.cwd().name == "notebooks" else Path.cwd() / "backend"
sys.path.insert(0, str(BACKEND))
load_dotenv(BACKEND / ".env")
MODEL = os.getenv("MODEL", "claude-sonnet-5")
print("Backend found:", BACKEND.exists(), "| key loaded:", bool(os.getenv("ANTHROPIC_API_KEY")), "| model:", MODEL)

## Step 1 · Say hello to Claude
A phone line to the brain. If it answers **Four**, everything is wired up.

In [ ]:
import anthropic

client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from the environment

def ask_claude(prompt: str) -> str:
    message = client.messages.create(
        model=MODEL, max_tokens=100, messages=[{"role": "user", "content": prompt}]
    )
    return message.content[0].text

print(ask_claude("What is 2 + 2? Answer in one word."))

## Step 2 · The toolbox
Claude only ever sees the **labels** (name, description, input schema). Our Python does the real work.
The calculator reads the expression as a syntax tree and only allows maths, so unlike `eval()` it can't run arbitrary code.

In [ ]:
from app.tools import CLIENT_TOOLS
from app.tools.calculator import calculate, CalculatorError

for t in CLIENT_TOOLS:
    print(f"{t['name']:15} {t['description'][:70]}...")

print("\n10000 * (1 + 0.07) ** 10 =", calculate("10000 * (1 + 0.07) ** 10"))
try:
    calculate("__import__('os').system('ls')")
except CalculatorError as e:
    print("Blocked:", e)

## Step 3 · The ReAct loop, minimal version
Think, then act, then observe, and repeat until `stop_reason == "end_turn"`. Here it is with the calculator only.
The production loop in `app/agent.py` adds streaming, more tools and `pause_turn` handling.

In [ ]:
CALC_ONLY = [t for t in CLIENT_TOOLS if t["name"] == "calculator"]

def run_agent(question: str, max_iterations: int = 6) -> str:
    messages = [{"role": "user", "content": question}]
    for step in range(1, max_iterations + 1):
        response = client.messages.create(model=MODEL, max_tokens=1024, tools=CALC_ONLY, messages=messages)
        print(f"[step {step}] stop_reason = {response.stop_reason}")
        if response.stop_reason == "end_turn":
            return "".join(b.text for b in response.content if b.type == "text")
        elif response.stop_reason == "tool_use":
            messages.append({"role": "assistant", "content": response.content})
            results = []
            for block in response.content:
                if block.type == "tool_use":
                    output = calculate(block.input["expression"])
                    print(f"   ACT  {block.name}({block.input}) -> OBSERVE {output}")
                    results.append({"type": "tool_result", "tool_use_id": block.id, "content": output})
            messages.append({"role": "user", "content": results})
        else:
            return f"Stopped: {response.stop_reason}"
    return "Hit the step limit."

print(run_agent("What is the square root of 144 multiplied by the number of days in a leap year?"))

## Step 4 · The real agent: every tool, streamed
This is the same `Agent` class the website uses. Jupyter can `await` directly, so we stream each event as it happens.

In [ ]:
import httpx
from app.agent import Agent
from app.config import get_settings
from app.llm import build_anthropic
from app.store import MemoryStore
from app.tools import ToolContext, build_tool_specs

settings = get_settings()
agent_client = build_anthropic(settings)
tools = build_tool_specs(settings.enable_web_search, settings.web_search_max_uses)
http = httpx.AsyncClient(timeout=8.0)
ctx = ToolContext(http=http, store=MemoryStore())

async def ask(history: list, question: str) -> None:
    history.append({"role": "user", "content": question})
    agent = Agent(agent_client, settings.model, tools, settings.max_tokens, settings.max_iterations)
    async for event in agent.run(history, ctx):
        if event["type"] == "text":
            print(event["delta"], end="")
        elif event["type"] == "tool_call":
            print(f"\n  -> {event['name']}({json.dumps(event['input'])})")
        elif event["type"] == "tool_result":
            print(f"  <- {event['output'][:160]}\n")
        elif event["type"] == "done":
            print(f"\n[{event['iterations']} model calls, tools: {event['tools_used']}]")

await ask([], "What's the weather in Vilnius, and what time is it in Tokyo?")

## Step 5 · Memory
Memory is just the `history` list, sent back to Claude on every turn. The website stores it in Supabase, so it survives page refreshes.

In [ ]:
history = []
await ask(history, "My budget is $50,000. How much will I have after 5 years at 7% a year?")
print("\n" + "-" * 60 + "\n")
await ask(history, "Now do the same calculation for 10 years.")
print(f"\nThe history now holds {len(history)} messages.")

In [ ]:
await http.aclose()  # tidy up the HTTP client